In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, lag, avg, dayofweek, month


class GoldFeatureEngineering:

    def __init__(self, spark, gold_table, feature_table):
        self.spark = spark
        self.gold_table = gold_table
        self.feature_table = feature_table
        self.df = None
        self.feature_df = None

    def read_gold(self):
        self.df = self.spark.table(self.gold_table)

    def add_calendar_features(self):
        self.df = (
            self.df
            .withColumn("day_of_week", dayofweek(col("transaction_date")))
            .withColumn("month", month(col("transaction_date")))
        )

    def add_lag_and_rolling_features(self):
        # Partition by product + city so lag/rolling never leaks across different series
        window_spec = Window.partitionBy("product_name", "destination_city").orderBy("transaction_date")

        # Rolling 7-row window: current row + 6 preceding rows within same partition
        rolling_window_spec = (
            Window.partitionBy("product_name", "destination_city")
            .orderBy("transaction_date")
            .rowsBetween(-6, 0)
        )

        self.df = (
            self.df
            .withColumn("demand_lag_1", lag("total_demand_quantity", 1).over(window_spec))
            .withColumn("demand_lag_7", lag("total_demand_quantity", 7).over(window_spec))
            .withColumn("rolling_avg_7", avg("total_demand_quantity").over(rolling_window_spec))
        )

    def select_final_columns(self):
        self.feature_df = self.df.select(
            "transaction_date",
            "product_name",
            "destination_city",
            "total_demand_quantity",
            "avg_unit_price_usd",
            "total_available_inventory",
            "day_of_week",
            "month",
            "demand_lag_1",
            "demand_lag_7",
            "rolling_avg_7"
        )

    def write_feature_table(self):
        self.feature_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(self.feature_table)
        print("Feature table created: " + self.feature_table)

    def run(self):
        self.read_gold()
        self.add_calendar_features()
        self.add_lag_and_rolling_features()
        self.select_final_columns()
        self.write_feature_table()


# ---------------------------
# Notebook usage
# ---------------------------
feature_pipeline = GoldFeatureEngineering(
    spark=spark,
    gold_table="oil_gas_mlops.gold.gold_supply_chain",
    feature_table="oil_gas_mlops.gold.feature_table"
)

feature_pipeline.run()